# 05 · Excel de entrega

Genera el Excel con el formato de Martin a partir de los JSON Lines de cada corrida. Se puede generar en cualquier momento; **nunca se edita a mano** (se vuelve a generar).

- Un solo Excel con todas las corridas de la lista (filas una debajo de otra). Error si un registro está en dos corridas.
- **Solo anotación manual**: nada de los modelos va al Excel.
- Hojas: `legend`, `map`, `identification`, una hoja de registros por corrida (`A1`, `A2`, según `sheet_name` del config), `traceability`, `excluded`.
- `site` y `season` en blanco (pendientes: sectores exactos).
- Se guarda en `MyDrive/sepia-inaturalist/exports/`.

| Notebook | Para qué |
|---|---|
| `01_descarga_y_modelos` | bajar registros de iNaturalist y correr el triaje y los 4 prompts |
| `02_anotacion` | anotar: asistida (elegir el mejor valor de los modelos), revisión con letra y recuadro, o a ciegas |
| `03_revision_y_metricas` | revisión interna: ranking de prompts, moda, métricas, persona vs modelos, duplicados |
| `04_revision_con_martin` | lo que se le muestra a Martin: qué hicimos, resultados, fotos, preguntas y el Excel |
| `05_entrega_excel` | generar y revisar el Excel de entrega |

La versión oficial de cada notebook está en GitHub (`notebooks/`, sin resultados). Para trabajar: abrirlo desde GitHub en Colab, *Archivo → Guardar una copia en Drive* y moverla a `MyDrive/sepia-inaturalist/notebooks/` (con la fecha en el nombre).

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline

sys.path.insert(0, '/content/sepia-inaturalist/src')
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Los 4 prompts que se usan desde el 4 nov (detalle en docs/prompt_versions.md)
VERSIONS = ['photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
ANNOTATOR = 'HZ'

In [ ]:
# Corridas (cada una en su carpeta de Drive). A1: anotacion a ciegas. A2: anotacion asistida.
cfg_a1 = sepia.load_config('configs/pilot_a1.yaml')
cfg_a2 = sepia.load_config('configs/a2_test.yaml')
RUNS = [cfg_a1, cfg_a2]

## 1. Control antes de exportar

In [ ]:
from sepia.annotate import load_events, done_photos, work_queue
for c in RUNS:
    ev = load_events(c)
    q = work_queue(c)
    done = done_photos(ev)
    pending = q[[(int(o), int(p)) not in done for o, p in zip(q['observation_id'], q['photo_id'])]]
    ind = ev['individual']
    print(f"{c.sheet_name or c.base_dir}: fotos anotadas {len(q) - len(pending)}/{len(q)} · "
          f"registros con fotos pendientes: {pending['observation_id'].nunique()} · "
          f"excluidos: {len(ev['exclude_record'])} · individuos: {len(ind)} · "
          f"sin recuadro: {int(ind['crop_box'].isna().sum()) if 'crop_box' in ind else len(ind)}")

## 2. Generar el Excel

In [ ]:
out = sepia.build_excel(RUNS, 'sepia_A1_A2.xlsx')   # sin nombre: lleva fecha y hora
print(out)

## 3. Vista rápida de las hojas

In [ ]:
import pandas as pd
xl = pd.ExcelFile(out)
print(xl.sheet_names)
for s in ('identification', 'A1'):
    if s in xl.sheet_names:
        display(xl.parse(s).head())